In [13]:
import os
import gzip
import json
import pandas as pd
from pathlib import Path
from collections.abc import Iterable, Iterator

# for agent


In [2]:
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "archive").is_dir())
ARCHIVE_DIR = REPO / "data" / "archive"  # gitignored download cache

In [3]:
def _resolve_paths(source: str | Path | Iterable[str | Path]) -> list[Path]:
    if isinstance(source, (str, Path)):
        p = Path(source)
        if p.is_dir():
            return sorted(p.glob("*.jsonl.gz"))
        return [p]
    return [Path(p) for p in source]

def _parse_filename(path: Path) -> tuple[str, str]:
    """``EARNINGS_RELEASE_2025Q3.jsonl.gz`` → ``("EARNINGS_RELEASE", "2025Q3")``."""
    stem = path.name.removesuffix(".jsonl.gz")
    event_type, _, quarter = stem.rpartition("_")
    return event_type, quarter

def read_jsonl_gz(path: str | Path) -> Iterator[dict]:
    """Yield each record from a gzip-JSONL file, skipping blank lines."""
    with gzip.open(path, "rt", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                yield json.loads(line)

def _load_one(path: Path) -> pd.DataFrame:
    df = pd.DataFrame(read_jsonl_gz(path))
    event_type, quarter = _parse_filename(path)
    # Add provenance from the filename, but never clobber a field the record
    # already carries. Insert at the front so it reads left-to-right.
    if "quarter" not in df.columns:
        df.insert(0, "quarter", quarter)
    if "event_type" not in df.columns:
        df.insert(0, "event_type", event_type)
    return df

def load_archive(source: str | Path | Iterable[str | Path]) -> pd.DataFrame:
    """Load one or more ``*.jsonl.gz`` archive files into a single DataFrame.

    ``source`` may be a directory (all ``*.jsonl.gz`` within it), a single file,
    or an iterable of paths. Provenance columns ``event_type`` and ``quarter``
    are added from each filename — unless a record already carries that field, in
    which case the record's own value is kept.
    """
    frames = [_load_one(p) for p in _resolve_paths(source)]
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

In [4]:
df_archive = load_archive(ARCHIVE_DIR)

In [ ]:
"""
[
  {
    "event_id": "123e4567-e89b-12d3-a456-426614174000",
    "event_type": "string",
    "timing_category": "SCHEDULED",
    "event_datetime": "2026-10-10T17:52:57.810Z",
    "knowledge_cutoff": "2026-10-10T17:52:57.810Z",
    "focal_assets": [
      {
        "identifier_type": "TICKER",
        "identifier_value": "AAPL"
      }
    ]
  }
]
"""

In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [69]:
row = df_archive.iloc[8500]
event_id = row["event_id"]
event_type = row["event_type"]
timing_category = row["timing_category"]
event_datetime = row["event_datetime"]
knowledge_cutoff = row["knowledge_cutoff"]
focal_assets = row["focal_assets"]
identifier_type = focal_assets[0]["identifier_type"]
identifier_value = focal_assets[0]["identifier_value"]

In [70]:
def safe_get(data, *keys):
    for key in keys:
        if not isinstance(data, dict):
            return None
        data = data.get(key)
    return data

In [ ]:
disclosure = row.get("disclosure") or {}
items = {
    d["id"]: d.get("content")
    for d in (disclosure.get("items") or [])
    if isinstance(d, dict) and d.get("id") is not None
}

earnings_call_date = disclosure.get("generated_at")
earnings_call_facts = items.get("earnings-call-facts")
earnings_preview = items.get("earnings-preview")
option_stats = items.get("option-implied-stats")

implied_earnings_volatility = safe_get(
    option_stats, "implied_earnings_volatility", "value"
)

implied_absolute_earnings_move = safe_get(
    option_stats, "implied_absolute_earnings_move", "value"
)

skew_25_delta = safe_get(
    option_stats, "skew_25_delta", "value"
)